<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.1_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic prompting

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama

In [1]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    base_url="https://ollama.com",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [3]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=llm)

start_time = time.time()

messages = [HumanMessage(content="What's the capital of the moon?")]
response = agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================

What's the capital of the moon?
================================== Ai Message ==================================

The Moon does not have a capital because it is not a country and has no government. It is a natural satellite with no permanent human inhabitants or sovereign states.
Time taken: 1.34s


In [4]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a capital city at the users request.
"""
scifi_agent = create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [HumanMessage(content="What's the capital of the moon?")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================

What's the capital of the moon?
================================== Ai Message ==================================

Since the Moon is currently a wilderness of basalt and regolith, its "capital" depends on which era of the future we are visiting. As a science fiction writer, I propose the most iconic seat of lunar power:

### **Selene Prime (The Glass Spire of Mare Tranquillitatis)**

**The Setting:**
Located in the heart of the *Sea of Tranquility*, Selene Prime is not a city of skyscrapers, but a city of "deep-scrapers." Because of the Moon’s lack of atmosphere and the constant threat of micrometeorite bombardment and solar radiation, the city is built primarily downward into the lunar crust.

**The Architecture:**
The only visible part of the capital from the surface is the **Apex Spire**—a needle-thin tower of reinforced lunar-glass and titanium that pierces the grey horizon. This spire acts as the city

## Few-shot examples

In [5]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""
scifi_agent = create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [HumanMessage(content="What's the capital of the moon?")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================

What's the capital of the moon?
================================== Ai Message ==================================

Scifi Writer: Selenopolis
Time taken: 0.65s


## Structured prompts

In [6]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries
"""
scifi_agent = create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [HumanMessage(content="What's the capital of the moon?")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================

What's the capital of the moon?
================================== Ai Message ==================================

Name: Selene Prime

Location: The Mare Tranquillitatis (Sea of Tranquility), Lunar Surface

Vibe: Sterile, monolithic, futuristic

Economy: Helium-3 mining, deep-space tourism, and lunar research
Time taken: 1.38s


## Structured output

In [9]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

system_prompt = """
You are a science fiction writer.
Create a capital city at the users request.
"""
agent = create_agent(
    model=llm,
    system_prompt=system_prompt,
    response_format=CapitalInfo.model_json_schema()
)

start_time = time.time()

messages = [HumanMessage(content="What is the capital of The Moon?")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

capital_info = response["structured_response"]
capital_name = capital_info["name"]
capital_location = capital_info["location"]
print(f"{capital_name} is a city located at {capital_location}")

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================

What is the capital of The Moon?
================================== Ai Message ==================================
Tool Calls:
  CapitalInfo (b94ef47c-8ea8-4e67-a1f9-956c4b6b2b65)
 Call ID: b94ef47c-8ea8-4e67-a1f9-956c4b6b2b65
  Args:
    economy: Helium-3 mining and low-gravity luxury tourism
    location: The Mare Tranquillitatis (Sea of Tranquility) basin
    name: Selene Prime
    vibe: A shimmering, subterranean metropolis of geodesic domes and neon-lit hydroponic gardens, where the silence of the vacuum meets the hum of high-tech industry.
================================= Tool Message =================================
Name: CapitalInfo

Returning structured response: {'economy': 'Helium-3 mining and low-gravity luxury tourism', 'location': 'The Mare Tranquillitatis (Sea of Tranquility) basin', 'name': 'Selene Prime', 'vibe': 'A shimmering, subterranean metropolis of geodesic domes and neon-lit hydro